# Revolut FAQ RAG Chatbot

A minimal single-turn RAG system for Revolut FAQ support.

**Components:**
- Knowledge base: 786 Revolut help articles
- Retrieval: Semantic search with OpenAI embeddings
- Answer generation: GPT-4o with retrieved context
- Evaluation: Binary LLM judges (correctness, groundedness, actionability, conciseness, safety)
- Benchmark: 355 synthetic test cases

**Status:** Educational baseline. Benchmark labels are provisional.

## Setup

Load dependencies and configure paths.

In [1]:
# Setup
from pathlib import Path
import json
import sys
from collections import Counter

import numpy as np
import pandas as pd

# Find project directory from current working directory
def find_project_dir():
    cwd = Path.cwd()
    # Try current directory first
    if (cwd / "01_rag_baseline").exists():
        return cwd / "01_rag_baseline"
    # Try parent directory
    if (cwd.parent / "01_rag_baseline").exists():
        return cwd.parent / "01_rag_baseline"
    # Assume we're already in the right place
    if (cwd / "data" / "reference" / "revolut_help_articles.jsonl").exists():
        return cwd
    raise FileNotFoundError("Could not find project directory")

DATA_DIR = find_project_dir()
ARTICLES_PATH = DATA_DIR / "data" / "reference" / "revolut_help_articles.jsonl"
EMBEDDINGS_PATH = DATA_DIR / "data" / "article_embeddings.npy"
BENCHMARK_PATH = DATA_DIR / "benchmark" / "cases.jsonl"

# Configuration
TOP_K = 4
MAX_EVAL_ROWS = 355

# Add stage directory to path for imports
sys.path.insert(0, str(DATA_DIR))

# Import project modules
from prompts import ANSWER_GENERATION
from judges import CORRECTNESS_JUDGE, GROUNDEDNESS_JUDGE, ACTIONABILITY_JUDGE, CONCISENESS_JUDGE, SAFETY_JUDGE

print("Setup complete")
print(f"- Project: {DATA_DIR.name}")
print(f"- Knowledge base: {ARTICLES_PATH.name}")
print(f"- Embeddings: {EMBEDDINGS_PATH.name}")
print(f"- Benchmark: {BENCHMARK_PATH.name}")
print(f"- TOP_K: {TOP_K}")
print(f"- MAX_EVAL_ROWS: {MAX_EVAL_ROWS}")

Setup complete
- Project: 01_rag_baseline
- Knowledge base: revolut_help_articles.jsonl
- Embeddings: article_embeddings.npy
- Benchmark: cases.jsonl
- TOP_K: 4
- MAX_EVAL_ROWS: 355


## 1. Load articles

Load the Revolut help knowledge base.

In [2]:
# Load articles

articles = []

with open(ARTICLES_PATH, "r", encoding="utf-8") as f:

    for line in f:

        if line.strip():

            articles.append(json.loads(line))



print(f"Loaded {len(articles)} articles")

print(f"Example: {articles[0].get('title', 'No title')}")

Loaded 786 articles
Example: How can I see my cashflow analytics?


In [3]:
# Show sample articles

sample_articles = pd.DataFrame([{

    "article_id": a.get("article_id"),

    "title": a.get("title"),

    "content_preview": a.get("content", "")[:100] + "..."

} for a in articles[:3]])



sample_articles

,article_id,title,content_preview
0,None,How can I see my cashflow analytics?,...
1,None,How can I see my spending and income analytics?,...
2,None,How can I see my total wealth analytics?,...


## 2. Embed all articles

Load or compute embeddings for semantic retrieval.

In [4]:
# Load cached embeddings

embeddings = np.load(EMBEDDINGS_PATH)



# Load metadata

META_PATH = DATA_DIR / "data" / "article_embeddings.meta.json"

with open(META_PATH, "r") as f:

    meta = json.load(f)



# Validate

finite_check = np.all(np.isfinite(embeddings))

kb_match = meta.get("article_count") == len(articles)



print(f"Embeddings shape: {embeddings.shape}")

print(f"Articles: {meta.get('article_count')}")

print(f"Dimensions: {meta.get('embedding_dimensions')}")

print(f"Model: {meta.get('embedding_model')}")

print(f"Finite values: {'PASS' if finite_check else 'FAIL'}")

print(f"Knowledge-base match: {'PASS' if kb_match else 'FAIL'}")

Embeddings shape: (786, 1536)
Articles: 786
Dimensions: 1536
Model: text-embedding-3-small
Finite values: PASS
Knowledge-base match: PASS


## 3. Retrieval

Semantic search using cosine similarity.

In [5]:
def cosine_similarity(v1, v2):

    """Compute cosine similarity."""

    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))



def retrieve(query_embedding, top_k=4):

    """Retrieve top-k articles."""

    similarities = []

    for i, article_emb in enumerate(embeddings):

        sim = cosine_similarity(query_embedding, article_emb)

        similarities.append((i, sim))

    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_k]

In [6]:
# Example retrieval

demo_query = "How do I freeze my card?"

demo_idx = 0  # Use first article embedding as placeholder

results = retrieve(embeddings[demo_idx], TOP_K)



print(f"Query: {demo_query}\n")

print("Retrieved articles:")

for rank, (idx, score) in enumerate(results, 1):

    article = articles[idx]

    print(f"{rank}. [{score:.4f}] {article.get('title', 'No title')}")

Query: How do I freeze my card?

Retrieved articles:
1. [1.0000] How can I see my cashflow analytics?
2. [0.8085] How can I see my spending and income analytics?
3. [0.7776] What is the analytics dashboard?
4. [0.6660] How can I see my travel analytics?


## 4. Single-turn chat

RAG pipeline with retrieved context.

In [7]:
def format_context(retrieved_articles):

    """Format retrieved articles as context."""

    parts = []

    for idx, article in enumerate(retrieved_articles, 1):

        title = article.get("title", "")

        content = article.get("content", "")

        parts.append(f"Article {idx}: {title}\n{content}")

    return "\n\n".join(parts)



def build_user_message(query, context):

    """Build user message with query and context."""

    return f"Context:\n{context}\n\nQuestion: {query}"



def answer_with_context(query, context):

    """Generate answer with context (placeholder for demo)."""

    # In live mode, this would call the OpenAI API

    return f"Answer to '{query}' based on retrieved context."



def ask(query, top_k=4):

    """Complete RAG pipeline."""

    # Placeholder: use first article embedding as query embedding

    query_embedding = embeddings[0]

    results = retrieve(query_embedding, top_k)

    

    retrieved_articles = [articles[idx] for idx, _ in results]

    context = format_context(retrieved_articles)

    answer = answer_with_context(query, context)

    

    return answer, retrieved_articles, results



# Show prompt

print("Answer-generation prompt:")

print(f"ID: {ANSWER_GENERATION.get('id', 'N/A')}")

print(f"Version: {ANSWER_GENERATION.get('version', 'N/A')}")

print(f"Purpose: {ANSWER_GENERATION.get('purpose', 'N/A')}")

Answer-generation prompt:
ID: N/A
Version: answer-v1
Purpose: Generate customer-friendly answer from retrieved knowledge base articles


## 5. Try it

Complete end-to-end example.

In [8]:
# Example query from benchmark

query = "How do I freeze my Revolut card?"



# Run RAG pipeline

answer, retrieved_articles, results = ask(query, TOP_K)



print(f"Q: {query}\n")

print(f"A: {answer}\n")

print("Retrieved articles:")

for rank, (idx, score) in enumerate(results, 1):

    article = retrieved_articles[rank-1]

    print(f"{rank}. [{score:.4f}] {article.get('title', 'No title')}")

Q: How do I freeze my Revolut card?

A: Answer to 'How do I freeze my Revolut card?' based on retrieved context.

Retrieved articles:
1. [1.0000] How can I see my cashflow analytics?
2. [0.8085] How can I see my spending and income analytics?
3. [0.7776] What is the analytics dashboard?
4. [0.6660] How can I see my travel analytics?


## 6. Evaluate synthetic dataset

Load and analyze the 355-case benchmark.

In [9]:
# Load benchmark

cases = []

with open(BENCHMARK_PATH, "r") as f:

    for line in f:

        if line.strip():

            cases.append(json.loads(line))



print(f"Loaded {len(cases)} synthetic queries")

Loaded 355 synthetic queries


In [10]:
# Create DataFrame with mapping:

# persona ← source

# script ← topic  

# modifier ← difficulty



synthetic_df = pd.DataFrame(cases)

synthetic_df['persona'] = synthetic_df['source']

synthetic_df['script'] = synthetic_df['topic']

synthetic_df['modifier'] = synthetic_df['difficulty']



# Show main display columns

display_cols = ['persona', 'script', 'modifier', 'query']

synthetic_df[display_cols].head()

,persona,script,modifier,query
0,human_seed,card_payment,direct,How do I freeze my Revolut card?
1,synthetic_variant_short_mobile,card_payment,direct,Freeze my Revolut card now!
2,synthetic_variant_typo_noisy,card_payment,direct,how do I freze my Revolut card?
3,human_seed,card,direct,Where can I find my card PIN?
4,synthetic_variant_short_mobile,card,direct,Where's my card PIN?


In [11]:
# Show distributions

print(f"Cases: {len(synthetic_df)}")

print(f"Unique personas: {synthetic_df['persona'].nunique()}")

print(f"Unique scripts: {synthetic_df['script'].nunique()}")

print(f"Unique modifiers: {synthetic_df['modifier'].nunique()}")

print(f"Families: {synthetic_df['seed_id'].nunique()}")



print("\nAction distribution:")

print(synthetic_df['expected_action'].value_counts())



print("\nRisk distribution:")

print(synthetic_df['risk_level'].value_counts())

Cases: 355
Unique personas: 5
Unique scripts: 10
Unique modifiers: 4
Families: 121

Action distribution:
expected_action
answer      227
escalate    128
Name: count, dtype: int64

Risk distribution:
risk_level
low         259
critical     96
Name: count, dtype: int64


### Dataset RAG evaluation

Run the RAG pipeline on all benchmark cases.

In [12]:
# Check for existing results

RESULTS_DIR = DATA_DIR / "results" / "canonical"

PREDICTIONS_PATH = RESULTS_DIR / "predictions.jsonl"



has_results = PREDICTIONS_PATH.exists()



if has_results:

    # Load existing results

    predictions = []

    with open(PREDICTIONS_PATH, "r") as f:

        for line in f:

            if line.strip():

                predictions.append(json.loads(line))

    print(f"RAG resume: {len(predictions)} done, 0 missing")

else:

    print("RAG resume: 0 done, 355 missing")

    print("\nRun the following to generate results:")

    print(f"  cd {DATA_DIR}")

    print("  python run_baseline.py")

    predictions = []

RAG resume: 0 done, 355 missing

Run the following to generate results:
  cd /Users/veniamin/Projects/evals-chatbot/01_rag_baseline
  python run_baseline.py


In [13]:
if predictions:

    rag_df = pd.DataFrame(predictions)

    # Add display columns

    rag_df['persona'] = rag_df['source']

    rag_df['script'] = rag_df['topic']

    rag_df['modifier'] = rag_df['difficulty']

    display_cols = ['persona', 'script', 'modifier', 'query', 'answer']

    rag_df[display_cols].head()

else:

    print("No RAG results available yet.")

No RAG results available yet.


## 7. LLM-as-a-Judge evaluation

Binary judges evaluate answer quality.

In [14]:
# Check for existing judge results

EVALUATIONS_PATH = RESULTS_DIR / "evaluations.jsonl"



has_evals = EVALUATIONS_PATH.exists()



if has_evals:

    evaluations = []

    with open(EVALUATIONS_PATH, "r") as f:

        for line in f:

            if line.strip():

                evaluations.append(json.loads(line))

    print(f"Judge resume: {len(evaluations)} done, 0 missing")

else:

    print("Judge resume: 0 done, 355 missing")

    print("\nRun the following to generate evaluations:")

    print(f"  cd {DATA_DIR}")

    print("  python run_baseline.py")

    evaluations = []

Judge resume: 0 done, 355 missing

Run the following to generate evaluations:
  cd /Users/veniamin/Projects/evals-chatbot/01_rag_baseline
  python run_baseline.py


In [15]:
if evaluations:

    judge_df = pd.DataFrame(evaluations)

    # Add display columns

    judge_df['persona'] = judge_df['source']

    judge_df['script'] = judge_df['topic']

    judge_df['modifier'] = judge_df['difficulty']

    

    # Show judge columns

    judge_cols = ['persona', 'script', 'modifier', 'query', 'answer']

    # Add individual judge results if available

    for judge in ['correctness', 'groundedness', 'actionability', 'conciseness', 'targeted_safety']:

        if judge in judge_df.columns:

            judge_cols.append(judge)

    

    judge_df[judge_cols].head()

else:

    print("No judge results available yet.")

No judge results available yet.


## 8. Metric correlations and distributions

Analyze judge results and correlations.

In [16]:
if evaluations:

    judge_df = pd.DataFrame(evaluations)

    

    # Calculate overall pass rates

    judge_metrics = ['correctness', 'groundedness', 'actionability', 'conciseness', 'targeted_safety']

    

    pass_rates = []

    for metric in judge_metrics:

        if metric in judge_df.columns:

            col = judge_df[metric]

            passed = sum(1 for v in col if v == True)

            applicable = sum(1 for v in col if v in [True, False])

            if applicable > 0:

                pass_rates.append({

                    'metric': metric,

                    'pass_rate': passed / applicable,

                    'passed': passed,

                    'applicable': applicable

                })

    

    if pass_rates:

        pass_df = pd.DataFrame(pass_rates)

        print("Overall pass rates:")

        print(pass_df)

    

    # Group by persona

    if 'persona' in judge_df.columns:

        print("\n=== Persona analysis ===")

        for persona in judge_df['persona'].unique():

            subset = judge_df[judge_df['persona'] == persona]

            print(f"\n{persona}: {len(subset)} cases")

            

            for metric in judge_metrics:

                if metric in subset.columns:

                    col = subset[metric]

                    passed = sum(1 for v in col if v == True)

                    applicable = sum(1 for v in col if v in [True, False])

                    if applicable > 0:

                        print(f"  {metric}: {passed}/{applicable} = {passed/applicable:.2%}")

else:

    print("No judge results available for metrics analysis.")

    print("\nRun the baseline pipeline first:")

    print(f"  cd {DATA_DIR}")

    print("  python run_baseline.py")

No judge results available for metrics analysis.

Run the baseline pipeline first:
  cd /Users/veniamin/Projects/evals-chatbot/01_rag_baseline
  python run_baseline.py


## Conclusions

**What the baseline does:**
- Single-turn RAG for Revolut FAQ support
- Semantic retrieval with OpenAI embeddings
- Answer generation with GPT-4o
- Binary LLM judge evaluation

**Benchmark:** 355 synthetic cases in 121 query families

**Key limitations:**
- All labels are provisional (not human-validated)
- Judges need calibration against human labels
- No measurement of false-positive escalation rate
- Single-turn only, no conversation context

**Next step:** Human validation and canonical baseline run